<a href="https://colab.research.google.com/github/jacobwechuli/mypython/blob/main/Wechuli_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q llama-index llama-index-embeddings-huggingface pymupdf
!pip install -q nest_asyncio

In [9]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, Document, Settings, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
import fitz  # PyMuPDF
import time

# Disable LLM - we're only comparing embedding models for retrieval
Settings.llm = None

LLM is explicitly disabled. Using MockLLM.


In [10]:
pdf_path = "/content/sample-sdf-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the pharmaceutical document.")

Extracted 617 words from the pharmaceutical document.


In [11]:
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

Created 73 chunks from the document.


In [12]:
embedding_models = {
    "MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small-en": "BAAI/bge-small-en-v1.5",
    "E5-small-v2": "intfloat/e5-small-v2"
}

# Pharmaceutical query about the SDF document
query = "What are the storage conditions for this product?"

results = {}

for model_name, model_path in embedding_models.items():
    print(f"\n{'='*60}")
    print(f"Testing Embedding Model: {model_name}")
    print(f"{'='*60}")

    # Configure the embedding model
    embed_model = HuggingFaceEmbedding(model_name=model_path)
    Settings.embed_model = embed_model

    # Build a fresh index with this embedding model
    # This ensures documents are embedded with the same model as queries
    start_time = time.time()
    index = VectorStoreIndex(nodes)

    retriever = index.as_retriever(similarity_top_k=2)
    query_engine = RetrieverQueryEngine.from_args(retriever=retriever)

    # Run the query
    response = query_engine.query(query)
    end_time = time.time()

    # Store results
    results[model_name] = {
        "response": str(response),
        "time": round(end_time - start_time, 2)
    }

    print(f"Index build + retrieval time: {results[model_name]['time']} seconds")


Testing Embedding Model: MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index build + retrieval time: 4.41 seconds

Testing Embedding Model: BGE-small-en


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index build + retrieval time: 7.25 seconds

Testing Embedding Model: E5-small-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index build + retrieval time: 7.19 seconds


In [13]:
print("\n" + "="*70)
print("EMBEDDING MODEL COMPARISON RESULTS")
print("="*70)

for model, result in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"{'='*60}")
    print(f"Index Build + Retrieval Time: {result['time']} seconds")
    print(f"\nRetrieved Context:")
    print("-" * 40)
    print(result['response'])
    print()

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("\nModel Performance Ranking (by total time):")
sorted_results = sorted(results.items(), key=lambda x: x[1]['time'])
for i, (model, result) in enumerate(sorted_results, 1):
    print(f"  {i}. {model}: {result['time']}s")


EMBEDDING MODEL COMPARISON RESULTS

Model: MiniLM-L6-v2
Index Build + Retrieval Time: 4.41 seconds

Retrieved Context:
----------------------------------------
Context information is below.
---------------------
MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,

com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.
---------------------
Given the context information and not prior knowledge, answer the query.
Query: What are the storage conditions for this product?
Answer: 


Model: BGE-small-en
Index Build + Retrieval Time: 7.25 seconds

Retrieved Context:
----------------------------------------
Context information is below.
---------------------
MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,

including 